# 03.1 Simple Chromatography: Exercises

## Contents

**Tools:** CADET-Process (Python).

**Python used:** Component-wise lists, event calls, parameter changes and simulation.

**Practises:** setting up `StericMassAction` binding, assembling a load–wash–elute method and comparing gradient duration at fixed transport and binding.

1. Set up the SMA binding model
2. Complete the method
3. Compare a shorter gradient

```{tip}
Run code cells from top to bottom with **Shift+Enter**. After entering, pasting or changing code, run that cell again to apply it, then rerun any later cells that use its results. Setup cells may show no output; they create objects for later cells. Fill empty answer cells as you go.
```

The goal is to recover **Protein A separately from Protein B**. The column and SMA binding parameters are the same as in the lesson; change the inlet method while keeping transport and binding fixed.

## Starter process

```{figure} resources/separation_system.svg
:width: 50%
```

The supplied process contains **Salt, Protein A, Protein B**, an inlet at **6 mL/min**, the **10 × 2 cm** lumped column and an outlet. Its cycle time is **63 min**. The column liquid is equilibrated at **50 mM salt**. The column has no binding model and the process has no events yet.

```{tip}
Run the cell below once with **Shift+Enter** to load the prebuilt model for this exercise. The [setup script](resources/chromatography_setup.py) contains the complete model construction.
```

In [ ]:
%matplotlib inline
# Load a fresh column with the lesson's transport and salt-dependent binding parameters.
%run resources/chromatography_setup.py
from CADETProcess.simulator import Cadet

exercise_column, exercise_process = separation_model("lwe_exercise", with_binding=False)
exercise_process.cycle_time = 63 * 60  # Convert minutes to seconds

## Exercise 1: set up the SMA binding model

Create a `StericMassAction` binding model for `exercise_process.component_system`, set the values from the table, assign it to `exercise_column.binding_model` and start with unloaded resin: `exercise_column.q = [225, 0, 0]`. Lists follow **Salt, Protein A, Protein B**.

| Attribute | Value |
|---|---|
| `is_kinetic` | `True` |
| `adsorption_rate` | `[0.0, 0.3, 0.2]` |
| `desorption_rate` | `[0.0, 1.5, 1.5]` |
| `characteristic_charge` | `[1, 7, 5]` |
| `steric_factor` | `[0, 50, 50]` |
| `capacity` | `225` |
| `reference_liquid_phase_conc` | `300` |
| `reference_solid_phase_conc` | `225` |

```{tip}
The binding model is created with `StericMassAction(component_system, "SMA")` from `CADETProcess.processModel`; the lesson **03.1 Simple Chromatography** shows the same setup. See also [`StericMassAction` in CADET-Process](https://cadet-process.readthedocs.io/en/latest/reference/generated/CADETProcess.processModel.StericMassAction.html).
```

In [ ]:
# SMA binding model
from CADETProcess.processModel import StericMassAction

binding = StericMassAction(exercise_process.component_system, "SMA")
binding.is_kinetic = True
binding.adsorption_rate = [0.0, 0.3, 0.2]
binding.desorption_rate = [0.0, 1.5, 1.5]
binding.characteristic_charge = [1, 7, 5]
binding.steric_factor = [0, 50, 50]
binding.capacity = 225
binding.reference_liquid_phase_conc = 300
binding.reference_solid_phase_conc = 225
exercise_column.binding_model = binding
exercise_column.q = [225, 0, 0]

```{dropdown} Answer
The first entry of each list belongs to salt. Its `characteristic_charge` is the salt valence **1**; its rates and steric factor are placeholders (**0**). `exercise_column.q = [225, 0, 0]` puts the whole ionic capacity on salt: no protein is bound at the start.
```

## Exercise 2: complete the method

**Load** Protein A at **0.08 mM** and Protein B at **0.12 mM** in **50 mM salt** for **3 min**. **Wash** at **50 mM salt**, without protein, for **10 min**. Then **elute** by increasing salt from **50 to 500 mM over 40 min**. Finish with **10 min final wash** at **500 mM salt**, without protein.

Add the four events to `exercise_process` in the answer cell. Parameter lists follow **Salt, Protein A, Protein B**. Convert minutes to **seconds** in the event calls. The gradient slope is (end salt − start salt) / gradient duration **in seconds**. Before simulating, predict which protein will elute first.

In [ ]:
# Load, wash, elute and final wash events
salt_slope = (500 - 50) / (40 * 60)
exercise_process.add_event("load", "flow_sheet.inlet.c", [50, 0.08, 0.12], 0)
exercise_process.add_event("wash", "flow_sheet.inlet.c", [50, 0, 0], 3 * 60)
exercise_process.add_event("elute", "flow_sheet.inlet.c", [[50, salt_slope], [0], [0]], 13 * 60)
exercise_process.add_event("final_wash", "flow_sheet.inlet.c", [500, 0, 0], 53 * 60)

In [ ]:
# Run the completed method and show its inlet profile and outlet chromatogram.
simulator = Cadet()
original_results = simulator.simulate(exercise_process)
_ = plot_lwe(original_results)

```{dropdown} Interpretation
Protein B elutes first in this model. The proteins entered together; their different binding parameters produce different retention during the salt gradient. Peak heights also reflect the different feed concentrations, so height alone does not measure binding strength.
```

## Exercise 3: compare a shorter gradient

Shorten elution from **40 to 20 min**, keeping its **13 min** start, the **50 and 500 mM** salt endpoints and the **10 min** final wash. Change `exercise_process.elute.state`, `exercise_process.final_wash.time` and `exercise_process.cycle_time` in the answer cell.

Predict how the time between the protein peaks and the total method time will change. The plotting cell retains the original results for comparison.

In [ ]:
# Shorter gradient: elution state, final wash and cycle time
salt_slope = (500 - 50) / (20 * 60)
exercise_process.elute.state = [[50, salt_slope], [0], [0]]
exercise_process.final_wash.time = 33 * 60
exercise_process.cycle_time = 43 * 60

In [ ]:
# Simulate the changed method and compare both outlet chromatograms on identical axes.
changed_results = simulator.simulate(exercise_process)
_ = plot_gradient_comparison(
    original_results, changed_results, "40 min gradient", "20 min gradient"
)

```{dropdown} Interpretation
The shorter method uses 20 min less time at the same flow rate, but gives less time between the protein peaks. Decide whether that is useful by considering the collection window and the purity required; a faster method is not automatically a better separation.
```